In [1]:
# cell1
# config
!pip install torch matplotlib -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import re
import os
import random
import math
from IPython.display import FileLink
import pandas as pd
from IPython.display import display, HTML
from collections import OrderedDict, Counter
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

# ========== 路径与全局常量 ==========
SAVE_DIR = "/kaggle/working/"
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "best_model.pth")
LOSS_CURVE_PATH = os.path.join(SAVE_DIR, "loss_curve.png")

DATA_PATH = "/kaggle/input/datasets/zn1350/chinese-poem/chinese_poem.txt"

# 读取用户输入的首句
user_input_path = "/kaggle/input/datasets/zn1350/chinese-poem/user_input_firstline.txt"
first_line = "春风又绿江南岸"

# 读取藏头字文件
user_input_acrostic = "/kaggle/input/datasets/zn1350/chinese-poem/user_input_acrostic.txt"
default_heads = ["春", "花", "秋", "月"]

MIN_SENTENCE_LEN = 7
MAX_SENTENCE_NUM = 4
BATCH_SIZE = 64
MAX_SEQ_LEN = 30

# 模型超参
EMBED_DIM = 256
NUM_HEADS = 8
NUM_LAYERS = 6
FFN_DIM = 1024
DROPOUT = 0.3

LR = 5e-4                     # 降低初始学习率
WEIGHT_DECAY = 0.01
EPOCHS = 50                   # 可自行调整
CLIP = 1.0
WARMUP_STEPS = 400

# 词汇表限制
MAX_VOCAB_SIZE = 3000         # 只保留高频字符

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print("CUDA available:", torch.cuda.is_available())

# 特殊标记
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
BOS_TOKEN = "<BOS>"
EOS_TOKEN = "<EOS>"

# 采样默认值
TEMPERATURE = 0.7
TOP_K = 8
TOP_P = 0.9
REPETITION_PENALTY = 1.2

demo_temp = 0.7
demo_topk = 8
demo_topp = 0.9
demo_repp = 1.2

ERROR: Could not find a version that satisfies the requirement cuda-bindings==12.9.4; platform_system == "Linux" (from torch) (from versions: none)
ERROR: No matching distribution found for cuda-bindings==12.9.4; platform_system == "Linux"
Using device: cuda
CUDA available: True


In [2]:
# cell2
# data_loader with vocabulary truncation
class PoemDataset(Dataset):
    def __init__(self, filepath, max_vocab=MAX_VOCAB_SIZE):
        self.poems = []
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
        lines = text.strip().split("\n")
        for line in lines:
            line = line.strip()
            if not line:
                continue
            sentences = re.split(r"[，。；！？]", line)
            sentences = [s.strip() for s in sentences if s.strip()]
            if len(sentences) == MAX_SENTENCE_NUM and all(len(s) == MIN_SENTENCE_LEN for s in sentences):
                self.poems.append(sentences)

        print(f"共加载 {len(self.poems)} 首符合要求的七言绝句")

        # 统计所有字符频率
        char_counter = Counter()
        for poem in self.poems:
            for sent in poem:
                char_counter.update(sent)

        # 取出现频率最高的 max_vocab 个字符
        most_common = [c for c, _ in char_counter.most_common(max_vocab)]
        self.idx2char = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN] + most_common
        self.char2idx = {c: i for i, c in enumerate(self.idx2char)}
        self.vocab_size = len(self.idx2char)

        print(f"词汇表大小: {self.vocab_size} (含特殊标记)")

        # 构建数据索引
        self.data = []
        for poem in self.poems:
            poem_str = "".join(poem)
            indices = [self.char2idx[BOS_TOKEN]] + \
                      [self.char2idx.get(c, self.char2idx[UNK_TOKEN]) for c in poem_str] + \
                      [self.char2idx[EOS_TOKEN]]
            self.data.append(torch.tensor(indices, dtype=torch.long))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    lengths = [len(x) for x in batch]
    max_len = max(lengths)
    padded = torch.full((len(batch), max_len), 0, dtype=torch.long)  # 0 is PAD_TOKEN index
    for i, seq in enumerate(batch):
        padded[i, :len(seq)] = seq
    return padded

def get_dataloader(filepath=DATA_PATH, batch_size=BATCH_SIZE, shuffle=True):
    dataset = PoemDataset(filepath)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)
    return dataloader, dataset.vocab_size, dataset.char2idx, dataset.idx2char

In [3]:
# cell3
# model: Transformer only (no LSTM) with Pre-LN and key_padding_mask
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class CharRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.pos_encoding = PositionalEncoding(EMBED_DIM)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS, enable_nested_tensor=False)

        self.fc = nn.Linear(EMBED_DIM, vocab_size)
        self.dropout = nn.Dropout(DROPOUT)

    def generate_causal_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1).to(DEVICE)
        return mask

    def forward(self, x):
        seq_len = x.size(1)
        key_padding_mask = (x == self.embedding.padding_idx)
        emb = self.dropout(self.embedding(x))
        emb = self.pos_encoding(emb)
        causal_mask = self.generate_causal_mask(seq_len)
        trans_out = self.transformer(
            emb,
            mask=causal_mask,
            src_key_padding_mask=key_padding_mask
        )
        logits = self.fc(self.dropout(trans_out))
        return logits

    # sample方法：强制生成28个有效汉字
    def sample(self, start_tokens, char2idx, idx2char, max_len=100,
               temperature=1.0, top_k=None, top_p=None,
               repetition_penalty=1.0, min_len=0, device=DEVICE):
        """
        强制生成 4句×7字 = 28 个有效字符（不含 BOS/EOS/PAD/UNK）
        生成满 28 字后自动补 EOS 结束
        """
        self.eval()
        with torch.no_grad():
            generated = list(start_tokens)   # 通常包含 BOS + 首句字符
            generated_set = set(start_tokens)
            special_ids = {char2idx[PAD_TOKEN], char2idx[UNK_TOKEN],
                           char2idx[BOS_TOKEN], char2idx[EOS_TOKEN]}

            # 目标：生成恰好 28 个有效汉字（不算 BOS 和 EOS）
            TARGET_CHARS = 28
            # 统计当前已有有效字符数（排除特殊标记）
            def count_valid(tokens):
                return sum(1 for t in tokens if t not in special_ids)

            # 继续生成，直到达到目标字符数
            while count_valid(generated) < TARGET_CHARS:
                input_seq = torch.tensor([generated], dtype=torch.long).to(device)
                logits = self(input_seq)
                logits = logits[0, -1, :] / temperature

                # 清理非法值（NaN/Inf）
                if torch.isnan(logits).any() or torch.isinf(logits).any():
                    logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)

                # 重复惩罚
                if repetition_penalty != 1.0:
                    for token_id in generated_set:
                        if token_id not in special_ids:
                            logits[token_id] /= repetition_penalty

                # top-k
                if top_k is not None and top_k > 0:
                    top_k = min(top_k, logits.size(-1))
                    top_vals, _ = torch.topk(logits, top_k)
                    logits[logits < top_vals[-1]] = -float('Inf')

                # top-p
                if top_p is not None and top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    indices_to_remove = sorted_indices[sorted_indices_to_remove]
                    logits[indices_to_remove] = -float('Inf')

                # 禁止连续两个相同 token
                if len(generated) >= 2 and generated[-1] == generated[-2]:
                    logits[generated[-1]] = -float('Inf')

                # 未满28个有效字符前禁止 EOS
                if count_valid(generated) < TARGET_CHARS:
                    logits[char2idx[EOS_TOKEN]] = -float('Inf')

                # 确保至少有一个有效的 token（避免全为 -inf）
                if torch.all(torch.isinf(logits)):
                    # 回退：使用均匀分布
                    logits = torch.zeros_like(logits)

                probs = F.softmax(logits, dim=-1)
                # 再次检查概率是否有效
                if torch.isnan(probs).any():
                    # 回退到均匀分布
                    probs = torch.ones_like(probs) / probs.size(-1)
                token = torch.multinomial(probs, 1).item()
                generated.append(token)
                generated_set.add(token)

            # 达到28个有效字符后强制补一个 EOS（如果还没有的话）
            if generated[-1] != char2idx[EOS_TOKEN]:
                generated.append(char2idx[EOS_TOKEN])

        return generated

In [4]:
# cell4
# 评估辅助函数（返回 loss 和 ppl）
def compute_metrics(model, dataloader, char2idx):
    model.eval()
    criterion = nn.CrossEntropyLoss(ignore_index=0, reduction='sum')
    total_loss = 0
    total_tokens = 0
    with torch.no_grad():
        for batch in dataloader:
            batch = batch.to(DEVICE)
            x = batch[:, :-1]
            y = batch[:, 1:]
            logits = model(x)
            loss = criterion(logits.reshape(-1, len(char2idx)), y.reshape(-1))
            total_loss += loss.item()
            total_tokens += (y != 0).sum().item()
    avg_loss = total_loss / total_tokens
    ppl = torch.exp(torch.tensor(avg_loss)).item()
    return avg_loss, ppl

def compute_ppl(model, dataloader, char2idx):
    _, ppl = compute_metrics(model, dataloader, char2idx)
    return ppl

def format_compliance(poem_str):
    lines = poem_str.strip().split('\n')
    if len(lines) != 4:
        return False
    for line in lines:
        if len(line.strip()) != 7:
            return False
    return True

In [5]:
# cell5
# train with proper train/val/test split

def train():
    # 先构建完整数据集（用于统计词表，不用于训练）
    full_dataset = PoemDataset(DATA_PATH)
    vocab_size = full_dataset.vocab_size
    char2idx = full_dataset.char2idx
    idx2char = full_dataset.idx2char
    
    total_size = len(full_dataset)
    train_ratio = 0.8
    val_ratio = 0.1
    test_ratio = 0.1
    
    train_size = int(total_size * train_ratio)
    val_size = int(total_size * val_ratio)
    test_size = total_size - train_size - val_size
    
    # 固定随机种子，保证划分可复现
    torch.manual_seed(42)
    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset, [train_size, val_size, test_size]
    )
    
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
    )
    
    print(f"数据集划分: 训练集 {train_size} 首, 验证集 {val_size} 首, 测试集 {test_size} 首")
    
    model = CharRNN(vocab_size).to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * len(train_loader)
    def lr_lambda(current_step):
        if current_step < WARMUP_STEPS:
            return float(current_step) / float(max(1, WARMUP_STEPS))
        progress = float(current_step - WARMUP_STEPS) / float(max(1, total_steps - WARMUP_STEPS))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    
    # 记录指标
    train_losses = []
    train_ppls = []
    test_losses = []
    test_ppls = []
    
    # 读取用户输入的首句（与划分无关）

    if os.path.exists(user_input_path):
        with open(user_input_path, "r", encoding="utf-8") as f:
            first_line = f.readline().strip()
            if first_line:
                first_line = first_line
                print(f"使用用户提供的首句: {first_line}")
            else:
                print(f"用户文件为空，使用后备首句: {first_line}")
    else:
        print(f"用户文件不存在，使用后备首句: {first_line}")
    
    # 初始化最佳验证困惑度
    best_val_ppl = float('inf')
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0
        total_tokens = 0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            x = batch[:, :-1]
            y = batch[:, 1:]
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() * y.numel()
            total_tokens += (y != 0).sum().item()
        
        avg_loss = total_loss / total_tokens
        train_ppl = torch.exp(torch.tensor(avg_loss)).item()
        train_losses.append(avg_loss)
        train_ppls.append(train_ppl)
        
        # 测试集评估（每个 epoch 都计算）
        test_loss, test_ppl = compute_metrics(model, test_loader, char2idx)
        test_losses.append(test_loss)
        test_ppls.append(test_ppl)
        
        # 验证集评估（用于选择最佳模型）
        val_ppl = compute_ppl(model, val_loader, char2idx)
        if val_ppl < best_val_ppl:
            best_val_ppl = val_ppl
            # 保存模型
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
                'char2idx': char2idx,
                'idx2char': idx2char,
                'vocab_size': vocab_size
            }, MODEL_SAVE_PATH)
            print(f"  -> 保存最佳模型 (loss {avg_loss:.4f}, val PPL {val_ppl:.2f})")
        
        print(f"Epoch {epoch:2d} | Train Loss: {avg_loss:.4f} | Train PPL: {train_ppl:.2f} | "
              f"Test Loss: {test_loss:.4f} | Test PPL: {test_ppl:.2f} | Val PPL: {val_ppl:.2f}")
        
        # 每 5 个 epoch 演示（使用用户首句）
        if epoch % 5 == 0:
            model.eval()
            with torch.no_grad():
                print(f"\n===== Epoch {epoch} 首句续写演示：{first_line} =====")
                print(f"当前验证集 PPL: {val_ppl:.2f}")
                
                start_seq = [char2idx[BOS_TOKEN]] + [char2idx.get(c, char2idx[UNK_TOKEN]) for c in first_line]
                if isinstance(model, nn.DataParallel):
                    generated = model.module.sample(start_seq, char2idx, idx2char, max_len=35,
                                                    temperature=demo_temp, top_k=demo_topk, top_p=demo_topp,
                                                    repetition_penalty=demo_repp, min_len=29)
                else:
                    generated = model.sample(start_seq, char2idx, idx2char, max_len=35,
                                             temperature=demo_temp, top_k=demo_topk, top_p=demo_topp,
                                             repetition_penalty=demo_repp, min_len=29)
                
                poem_chars = []
                for idx in generated:
                    if idx == char2idx["<EOS>"]:
                        break
                    if idx not in {char2idx[PAD_TOKEN], char2idx[UNK_TOKEN], char2idx[BOS_TOKEN]}:
                        poem_chars.append(idx2char[idx])
                poem = ''.join(poem_chars)
                
                if len(poem) >= 28:
                    formatted = '\n'.join([poem[i:i+7] for i in range(0, 28, 7)])
                else:
                    formatted = poem
                print(f"生成后三句（及第一句）:\n{formatted}")
                is_ok = format_compliance(formatted)
                print(f"格式合规: {'✓' if is_ok else '✗'}\n")
            model.train()
    
    # 训练结束后，在测试集上评估最终困惑度（可选，已有每个epoch的测试指标）
    final_test_loss, final_test_ppl = compute_metrics(model, test_loader, char2idx)
    print(f"\n最终测试集平均 Loss: {final_test_loss:.4f}, 困惑度 (PPL): {final_test_ppl:.2f}")
    
    # 绘制并保存四张曲线图
    epochs_range = range(1, EPOCHS+1)
    
    # 训练损失曲线
    plt.figure()
    plt.plot(epochs_range, train_losses, marker='o')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Curve")
    plt.grid(True)
    plt.savefig(os.path.join(SAVE_DIR, "train_loss.png"))
    plt.close()
    
    # 测试损失曲线
    plt.figure()
    plt.plot(epochs_range, test_losses, marker='s')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Test Loss Curve")
    plt.grid(True)
    plt.savefig(os.path.join(SAVE_DIR, "test_loss.png"))
    plt.close()
    
    # 训练困惑度曲线
    plt.figure()
    plt.plot(epochs_range, train_ppls, marker='o')
    plt.xlabel("Epoch")
    plt.ylabel("Perplexity")
    plt.title("Training Perplexity Curve")
    plt.grid(True)
    plt.savefig(os.path.join(SAVE_DIR, "train_ppl.png"))
    plt.close()
    
    # 测试困惑度曲线
    plt.figure()
    plt.plot(epochs_range, test_ppls, marker='s')
    plt.xlabel("Epoch")
    plt.ylabel("Perplexity")
    plt.title("Test Perplexity Curve")
    plt.grid(True)
    plt.savefig(os.path.join(SAVE_DIR, "test_ppl.png"))
    plt.close()
    
    print(f"四张曲线图已保存至 {SAVE_DIR}")
    from IPython.display import FileLink
    display(FileLink(os.path.join(SAVE_DIR, "train_loss.png")))
    display(FileLink(os.path.join(SAVE_DIR, "test_loss.png")))
    display(FileLink(os.path.join(SAVE_DIR, "train_ppl.png")))
    display(FileLink(os.path.join(SAVE_DIR, "test_ppl.png")))
    display(FileLink(MODEL_SAVE_PATH))
    
    return model, char2idx, idx2char

model, char2idx, idx2char = train()

共加载 139196 首符合要求的七言绝句
词汇表大小: 3004 (含特殊标记)
数据集划分: 训练集 111356 首, 验证集 13919 首, 测试集 13921 首
使用用户提供的首句: 春风又绿江南岸


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


  -> 保存最佳模型 (loss 6.5868, val PPL 338.26)
Epoch  1 | Train Loss: 6.5868 | Train PPL: 725.45 | Test Loss: 5.8253 | Test PPL: 338.75 | Val PPL: 338.26
  -> 保存最佳模型 (loss 5.8449, val PPL 223.17)
Epoch  2 | Train Loss: 5.8449 | Train PPL: 345.48 | Test Loss: 5.4109 | Test PPL: 223.83 | Val PPL: 223.17
  -> 保存最佳模型 (loss 5.5921, val PPL 190.68)
Epoch  3 | Train Loss: 5.5921 | Train PPL: 268.31 | Test Loss: 5.2533 | Test PPL: 191.19 | Val PPL: 190.68
  -> 保存最佳模型 (loss 5.4748, val PPL 173.99)
Epoch  4 | Train Loss: 5.4748 | Train PPL: 238.60 | Test Loss: 5.1606 | Test PPL: 174.26 | Val PPL: 173.99
  -> 保存最佳模型 (loss 5.3963, val PPL 163.48)
Epoch  5 | Train Loss: 5.3963 | Train PPL: 220.58 | Test Loss: 5.0980 | Test PPL: 163.69 | Val PPL: 163.48

===== Epoch 5 首句续写演示：春风又绿江南岸 =====
当前验证集 PPL: 163.48
生成后三句（及第一句）:
春风又绿江南岸
春色无情水北村
不识故人愁未尽
可怜春意在东西
格式合规: ✓

  -> 保存最佳模型 (loss 5.3289, val PPL 150.38)
Epoch  6 | Train Loss: 5.3289 | Train PPL: 206.21 | Test Loss: 5.0149 | Test PPL: 150.63 | Val PPL: 150.3

/kaggle/working/train_loss.png

/kaggle/working/test_loss.png

/kaggle/working/train_ppl.png

/kaggle/working/test_ppl.png

/kaggle/working/best_model.pth

In [6]:
# cell6
# ==================== 生成样本时显示每个序列的 loss 和 ppl（与训练时相同计算方法） ====================

def compute_sequence_loss_ppl(model, token_seq, char2idx):
    """
    计算一个 token 序列（列表，含 BOS 和 EOS）的 loss 和 ppl。
    使用与训练时完全相同的 nn.CrossEntropyLoss(ignore_index=0)。
    token_seq: list of int, 长度 L, 包含 BOS 和 EOS
    """
    model.eval()
    with torch.no_grad():
        input_ids = torch.tensor([token_seq], dtype=torch.long).to(DEVICE)  # (1, L)
        logits = model(input_ids)  # (1, L, vocab_size)
        targets = input_ids[:, 1:]  # (1, L-1)
        logits = logits[:, :-1, :]  # 对齐
        loss_fn = nn.CrossEntropyLoss(ignore_index=char2idx[PAD_TOKEN], reduction='mean')
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        ppl = torch.exp(loss).item()
    return loss.item(), ppl

def continue_first_line_with_loss_ppl(model, char2idx, idx2char, first_line, temperature, top_k, top_p=None):
    """返回 (格式化诗句, 完整token序列, loss, ppl)"""
    start_seq = [char2idx[BOS_TOKEN]] + [char2idx.get(c, char2idx[UNK_TOKEN]) for c in first_line]
    if isinstance(model, nn.DataParallel):
        generated = model.module.sample(start_seq, char2idx, idx2char, max_len=35,
                                        temperature=temperature, top_k=top_k, top_p=top_p,
                                        repetition_penalty=REPETITION_PENALTY, min_len=29)
    else:
        generated = model.sample(start_seq, char2idx, idx2char, max_len=35,
                                 temperature=temperature, top_k=top_k, top_p=top_p,
                                 repetition_penalty=REPETITION_PENALTY, min_len=29)
    # 去掉特殊标记，拼接成诗句字符串
    poem_chars = []
    for idx in generated:
        if idx == char2idx.get(EOS_TOKEN, -1):
            break
        if idx not in {char2idx[PAD_TOKEN], char2idx[UNK_TOKEN], char2idx[BOS_TOKEN]}:
            poem_chars.append(idx2char[idx])
    poem = ''.join(poem_chars)
    if len(poem) >= 28:
        formatted = '\n'.join([poem[i:i+7] for i in range(0, 28, 7)])
    else:
        formatted = poem
    # 计算 loss 和 ppl
    loss, ppl = compute_sequence_loss_ppl(model, generated, char2idx)
    return formatted, generated, loss, ppl

# 加载最佳模型
print("加载最佳模型...")
ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
vocab_size = ckpt['vocab_size']
char2idx = ckpt['char2idx']
idx2char = ckpt['idx2char']

state_dict = ckpt['model_state_dict']
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    if k.startswith('module.'):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

model = CharRNN(vocab_size).to(DEVICE)
model.load_state_dict(new_state_dict)
model.eval()
print("模型加载成功\n")

# 测试集整体 PPL
dataset = PoemDataset(DATA_PATH)
total = len(dataset)
test_size = max(1, int(total * 0.1))
indices = list(range(total))
random.shuffle(indices)
test_indices = indices[:test_size]
test_subset = torch.utils.data.Subset(dataset, test_indices)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False, collate_fn=collate_fn)
overall_ppl = compute_ppl(model, test_loader, char2idx)
print(f"测试集整体困惑度 (PPL): {overall_ppl:.2f}\n")

# 读取用户输入的首句

if os.path.exists(user_input_path):
    with open(user_input_path, "r", encoding="utf-8") as f:
        line = f.readline().strip()
        if line:
            first_line = line
            print(f"使用用户提供的首句: {first_line}")
        else:
            print(f"用户文件为空，使用后备首句: {first_line}")
else:
    print(f"用户文件不存在，使用后备首句: {first_line}")

# ========== 同一采样策略的 5 个随机样本（显示每个样本的 Loss 和 PPL） ==========
print("=" * 60)
print(f"【同一采样策略演示】默认温度 {TEMPERATURE}, top_k={TOP_K}, top_p={TOP_P}")
print(f"首句：{first_line}")
print("=" * 60)
for i in range(5):
    poem, full_seq, loss, ppl = continue_first_line_with_loss_ppl(
        model, char2idx, idx2char, first_line,
        TEMPERATURE, TOP_K, TOP_P
    )
    is_ok = format_compliance(poem)
    print(f"样本 {i+1}:")
    print(f"  续写诗句:\n{poem}")
    print(f"  Loss: {loss:.4f}, PPL: {ppl:.2f}")
    print(f"  格式合规: {'✓' if is_ok else '✗'}\n")

加载最佳模型...
模型加载成功

共加载 139196 首符合要求的七言绝句
词汇表大小: 3004 (含特殊标记)
测试集整体困惑度 (PPL): 83.84

使用用户提供的首句: 春风又绿江南岸
【同一采样策略演示】默认温度 0.7, top_k=8, top_p=0.9
首句：春风又绿江南岸
样本 1:
  续写诗句:
春风又绿江南岸
秋色初红海上山
今日西湖无限恨
只应惆怅在人间
  Loss: 2.6098, PPL: 13.60
  格式合规: ✓

样本 2:
  续写诗句:
春风又绿江南岸
秋月初明水北门
莫问东归何处去
一枝芳草几回魂
  Loss: 2.7137, PPL: 15.08
  格式合规: ✓

样本 3:
  续写诗句:
春风又绿江南岸
雨后秋青草满村
不是无限意可怜
憔悴更伤魂碑屋
  Loss: 7.3401, PPL: 1540.86
  格式合规: ✓

样本 4:
  续写诗句:
春风又绿江南岸
草木无情水上村
一片红尘无处觅
只应回首见青云
  Loss: 2.6045, PPL: 13.52
  格式合规: ✓

样本 5:
  续写诗句:
春风又绿江南岸
春草初黄野水村
无言归不得一樽
相对一尊存诘旨
  Loss: 7.0396, PPL: 1140.96
  格式合规: ✓



In [7]:
# cell7
# ============ 策略对比：20样本平均指标 + 单样本详细对比表 ============
print("加载最佳模型用于策略对比...")
ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
vocab_size = ckpt['vocab_size']
char2idx = ckpt['char2idx']
idx2char = ckpt['idx2char']

state_dict = ckpt['model_state_dict']
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    if k.startswith('module.'):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

model = CharRNN(vocab_size).to(DEVICE)
model.load_state_dict(new_state_dict)
model.eval()
print("最佳模型加载成功\n")


if os.path.exists(user_input_path):
    with open(user_input_path, "r", encoding="utf-8") as f:
        line = f.readline().strip()
        if line:
            first_line = line
            print(f"使用用户提供的首句: {first_line}")
        else:
            print(f"用户文件为空，使用后备首句: {first_line}")
else:
    print(f"用户文件不存在，使用后备首句: {first_line}")

# ========== 不同采样策略对比（每种策略生成 20 个样本） ==========
print("\n" + "=" * 60)
print("不同采样策略对比（每种策略生成 20 个样本）")
print("=" * 60)
strategies_20 = [
    {"name": "Greedy",         "temp": 1.0, "top_k": 1,   "top_p": None},
    {"name": "T=0.6 K=10",     "temp": 0.6, "top_k": 10,  "top_p": None},
    {"name": "T=0.8 K=20",     "temp": 0.8, "top_k": 20,  "top_p": None},
    {"name": "T=1.2 K=30",     "temp": 1.2, "top_k": 30,  "top_p": None},
    {"name": "T=0.7 P=0.9",    "temp": 0.7, "top_k": None,"top_p": 0.9},
]

for s in strategies_20:
    ok_count = 0
    losses = []
    ppls = []
    for _ in range(20):
        poem, full_seq, loss, ppl = continue_first_line_with_loss_ppl(
            model, char2idx, idx2char, first_line,
            s["temp"], s["top_k"], s["top_p"]
        )
        if format_compliance(poem):
            ok_count += 1
        losses.append(loss)
        ppls.append(ppl)
    avg_loss = np.mean(losses)
    std_loss = np.std(losses)
    avg_ppl = np.mean(ppls)
    std_ppl = np.std(ppls)
    print(f"{s['name']}:")
    print(f"  格式合规率: {ok_count/20:.2%}")
    print(f"  平均 Loss: {avg_loss:.4f} (±{std_loss:.4f})")
    print(f"  平均 PPL: {avg_ppl:.2f} (±{std_ppl:.2f})\n")

# ========== 详细策略对比表：每个策略下生成的诗句及其 loss 和 ppl ==========
print("\n" + "=" * 60)
print("首句续写各策略详细对比（固定随机种子42）")
print("=" * 60)

strategies_detail = [
    {"name": "温度0.5 无top_k",   "temperature": 0.5, "top_k": None, "top_p": None},
    {"name": "温度0.7 无top_k",   "temperature": 0.7, "top_k": None, "top_p": None},
    {"name": "温度1.0 无top_k",   "temperature": 1.0, "top_k": None, "top_p": None},
    {"name": "温度1.2 无top_k",   "temperature": 1.2, "top_k": None, "top_p": None},
    {"name": "温度0.8 top_k=5",   "temperature": 0.8, "top_k": 5,   "top_p": None},
    {"name": "温度0.8 top_k=10",  "temperature": 0.8, "top_k": 10,  "top_p": None},
    {"name": "温度0.8 top_p=0.9", "temperature": 0.8, "top_k": None, "top_p": 0.9},
]

rows = []
for i, s in enumerate(strategies_detail, 1):
    torch.manual_seed(42)   # 固定种子，使结果可复现
    poem, full_seq, loss, ppl = continue_first_line_with_loss_ppl(
        model, char2idx, idx2char, first_line,
        temperature=s["temperature"], top_k=s["top_k"], top_p=s["top_p"]
    )
    analysis = {
        "四句完整": "✓" if format_compliance(poem) else "✗",
        "每句七言": "✓" if all(len(line.strip())==7 for line in poem.strip().split('\n')) else "✗"
    }
    print(f"\n【策略{i}: {s['name']}】")
    print(poem)
    print(f"Loss: {loss:.4f}, PPL: {ppl:.2f}")
    print(f"格式: 四句完整={analysis['四句完整']}, 每句七言={analysis['每句七言']}")
    rows.append({
        "策略": s["name"],
        "生成续写": poem.replace('\n', ' | '),
        "Loss": f"{loss:.4f}",
        "PPL": f"{ppl:.2f}",
        "四句完整": analysis["四句完整"],
        "每句七言": analysis["每句七言"]
    })

print("\n" + "=" * 60)
print("各策略效果对比汇总表")
print("=" * 60)
df = pd.DataFrame(rows)
display(HTML(df.to_html(index=False)))

加载最佳模型用于策略对比...
最佳模型加载成功

使用用户提供的首句: 春风又绿江南岸

不同采样策略对比（每种策略生成 20 个样本）
Greedy:
  格式合规率: 100.00%
  平均 Loss: 2.4206 (±0.0000)
  平均 PPL: 11.25 (±0.00)

T=0.6 K=10:
  格式合规率: 100.00%
  平均 Loss: 3.2346 (±1.4140)
  平均 PPL: 265.98 (±1012.69)

T=0.8 K=20:
  格式合规率: 100.00%
  平均 Loss: 3.2673 (±0.8240)
  平均 PPL: 54.86 (±131.05)

T=1.2 K=30:
  格式合规率: 100.00%
  平均 Loss: 3.8163 (±0.9401)
  平均 PPL: 93.56 (±189.14)

T=0.7 P=0.9:
  格式合规率: 100.00%
  平均 Loss: 3.4156 (±0.9724)
  平均 PPL: 83.85 (±223.25)


首句续写各策略详细对比（固定随机种子42）

【策略1: 温度0.5 无top_k】
春风又绿江南岸
绿水无情草木秋
不是重门多少事
数声啼鸟自悠悠
Loss: 2.8035, PPL: 16.50
格式: 四句完整=✓, 每句七言=✓

【策略2: 温度0.7 无top_k】
春风又绿江南岸
绿涨无痕客裾有
此重门携酒去数
声乌鹊噪啼乌琶
Loss: 5.0655, PPL: 158.46
格式: 四句完整=✓, 每句七言=✓

【策略3: 温度1.0 无top_k】
春风又绿江南岸
绿涨无痕倚天有
此肠断何处听落
人情绪若离然琶
Loss: 5.5753, PPL: 263.84
格式: 四句完整=✓, 每句七言=✓

【策略4: 温度1.2 无top_k】
春风又绿江南岸
绿涨无痕倚天书
僻重门但温昼堂
筵少住若离篇郸
Loss: 6.7987, PPL: 896.67
格式: 四句完整=✓, 每句七言=✓

【策略5: 温度0.8 top_k=5】
春风又绿江南岸
柳絮无情草木阴
不是故人多少事
只应回首泪沾襟
Loss: 2.5468, PPL: 12.77
格式: 四句完整=✓, 每句七言=✓



策略,生成续写,Loss,PPL,四句完整,每句七言
温度0.5 无top_k,春风又绿江南岸 | 绿水无情草木秋 | 不是重门多少事 | 数声啼鸟自悠悠,2.8035,16.50,✓,✓
温度0.7 无top_k,春风又绿江南岸 | 绿涨无痕客裾有 | 此重门携酒去数 | 声乌鹊噪啼乌琶,5.0655,158.46,✓,✓
温度1.0 无top_k,春风又绿江南岸 | 绿涨无痕倚天有 | 此肠断何处听落 | 人情绪若离然琶,5.5753,263.84,✓,✓
温度1.2 无top_k,春风又绿江南岸 | 绿涨无痕倚天书 | 僻重门但温昼堂 | 筵少住若离篇郸,6.7987,896.67,✓,✓
温度0.8 top_k=5,春风又绿江南岸 | 柳絮无情草木阴 | 不是故人多少事 | 只应回首泪沾襟,2.5468,12.77,✓,✓
温度0.8 top_k=10,春风又绿江南岸 | 花落无情草木阴 | 不是故人多少事 | 故人相对两三心,2.7495,15.63,✓,✓
温度0.8 top_p=0.9,春风又绿江南岸 | 绿涨无痕客裾有 | 此重来携酒去数 | 声乌鸟共离居法,5.8473,346.29,✓,✓


In [8]:
# cell8
def generate_acrostic(model, head_chars, char2idx, idx2char,
                      temperature=0.8, top_k=10, top_p=None,
                      repetition_penalty=1.2, device=DEVICE):
    """
    生成藏头诗，强制每句首字为 head_chars，严格生成 4×7=28 个有效汉字，
    禁止生成任何特殊标记（PAD/UNK/EOS），最后手动添加 EOS。
    返回 (formatted_poem, token_sequence)
    """
    model.eval()
    assert len(head_chars) == 4, "藏头诗需要 4 个首字"

    # 检查所有首字是否在词表中，若缺失则用 UNK（但会引发后续断言）
    for ch in head_chars:
        if ch not in char2idx:
            raise ValueError(f"首字 '{ch}' 不在词汇表中，请更换或扩展词汇表。")

    special_ids = {char2idx[PAD_TOKEN], char2idx[UNK_TOKEN],
                   char2idx[BOS_TOKEN], char2idx[EOS_TOKEN]}
    # 用于采样时屏蔽特殊标记
    banned_ids = {char2idx[PAD_TOKEN], char2idx[UNK_TOKEN], char2idx[EOS_TOKEN]}

    generated = [char2idx[BOS_TOKEN]]   # 以 BOS 开头

    with torch.no_grad():
        for i, head_char in enumerate(head_chars):
            # ----- 强制添加本句首字 -----
            head_idx = char2idx[head_char]   # 此时保证存在
            generated.append(head_idx)

            # ----- 生成后续 6 个字 -----
            for pos in range(6):
                input_seq = torch.tensor([generated], dtype=torch.long).to(device)
                logits = model(input_seq)
                logits = logits[0, -1, :] / temperature

                # 清理 NaN/Inf
                if torch.isnan(logits).any() or torch.isinf(logits).any():
                    logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)

                # 重复惩罚（对已出现的字符降权，但不影响强制首字）
                if repetition_penalty != 1.0:
                    for token_id in set(generated):
                        if token_id not in special_ids:
                            logits[token_id] /= repetition_penalty

                # ----- 屏蔽所有特殊标记（严格禁止 PAD/UNK/EOS）-----
                for bid in banned_ids:
                    logits[bid] = -float('Inf')

                # top-k
                if top_k is not None and top_k > 0:
                    k = min(top_k, logits.size(-1))
                    top_vals, _ = torch.topk(logits, k)
                    logits[logits < top_vals[-1]] = -float('Inf')

                # top-p
                if top_p is not None and top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    indices_to_remove = sorted_indices[sorted_indices_to_remove]
                    logits[indices_to_remove] = -float('Inf')

                # 避免连续两个相同 token
                if len(generated) >= 2 and generated[-1] == generated[-2]:
                    logits[generated[-1]] = -float('Inf')

                # 确保至少有一个合法 token（避免全 -inf）
                if torch.all(torch.isinf(logits)):
                    # 回退：所有特殊标记已被屏蔽，此时全 -inf 只会发生在极端情况，使用均匀分布
                    logits = torch.zeros_like(logits)
                    # 再次屏蔽特殊标记
                    for bid in banned_ids:
                        logits[bid] = -float('Inf')

                probs = F.softmax(logits, dim=-1)
                if torch.isnan(probs).any():
                    probs = torch.ones_like(probs) / probs.size(-1)
                    for bid in banned_ids:
                        probs[bid] = 0.0

                next_token = torch.multinomial(probs, 1).item()
                generated.append(next_token)

        # 生成完 28 个有效字符后，手动添加 EOS
        generated.append(char2idx[EOS_TOKEN])

    # ----- 提取有效字符（直接跳过 BOS，取到 EOS 之前）-----
    # 因为生成过程中已禁止特殊标记，所以 generated[1:-1] 即为 28 个有效汉字
    token_seq = generated[1:-1]   # 去掉 BOS 和 EOS
    if len(token_seq) != 28:
        # 防御性处理：若不足 28 则用 '？' 补齐（但理论上不会发生）
        token_seq = token_seq + [char2idx.get('？', char2idx[UNK_TOKEN])] * (28 - len(token_seq))

    # 转换为字符串并格式化
    poem_chars = [idx2char[idx] for idx in token_seq]
    poem_str = ''.join(poem_chars[:28])
    formatted = '\n'.join([poem_str[i:i+7] for i in range(0, 28, 7)])

    # 返回格式化的诗句和包含 BOS/EOS 的完整序列（用于计算 loss/ppl）
    full_seq = [char2idx[BOS_TOKEN]] + token_seq + [char2idx[EOS_TOKEN]]
    return formatted, full_seq

In [9]:
# Cell 9
# 调用藏头诗生成函数，输出结果
print("\n" + "=" * 70)
print("藏头诗生成示例（使用最佳模型）")
print("=" * 70)

# 确保模型已加载（如果尚未加载，请先加载最佳模型）
if 'model' not in dir() or model is None:
    print("加载最佳模型...")
    ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
    vocab_size = ckpt['vocab_size']
    char2idx = ckpt['char2idx']
    idx2char = ckpt['idx2char']
    state_dict = ckpt['model_state_dict']
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    model = CharRNN(vocab_size).to(DEVICE)
    model.load_state_dict(new_state_dict)
    model.eval()
    print("模型加载成功\n")

# 读取藏头字（默认或从文件）
heads = default_heads.copy()
if os.path.exists(user_input_acrostic):
    with open(user_input_acrostic, "r", encoding="utf-8") as f:
        content = f.readline().strip()
        if content and len(content) >= 4:
            heads = list(content[:4])
            print(f"使用文件提供的藏头字：{' '.join(heads)}")
        else:
            print(f"文件内容不足4字，使用默认藏头字：{' '.join(default_heads)}")
else:
    print(f"藏头字文件不存在，使用默认藏头字：{' '.join(default_heads)}")

# 生成一首藏头诗（采用默认采样参数）
torch.manual_seed(42)  # 固定种子以便复现
poem, token_seq = generate_acrostic(
    model, heads, char2idx, idx2char,
    temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY
)

# 计算该序列的 Loss 和 PPL
loss, ppl = compute_sequence_loss_ppl(model, token_seq, char2idx)

print("\n生成的藏头诗：")
print(poem)
print(f"Loss: {loss:.4f}, PPL: {ppl:.2f}")

# 校验首字
lines = poem.strip().split('\n')
if len(lines) == 4:
    ok = all(lines[i][0] == heads[i] for i in range(4))
    print(f"首字校验: {'✓' if ok else '✗'}")
else:
    print(f"警告：生成的诗句行数为 {len(lines)}，非4行")


藏头诗生成示例（使用最佳模型）
使用文件提供的藏头字：春 花 秋 月

生成的藏头诗：
春风一夜满江干
花影萧萧月色寒
秋草满庭人未起
月明空锁暮云间
Loss: 2.7122, PPL: 15.06
首字校验: ✓


In [10]:
# Cell 10
# 藏头诗不同采样策略对比（20样本统计 + 固定种子详细对比）
print("\n" + "=" * 70)
print("藏头诗不同采样策略对比")
print("=" * 70)

# 确保模型已加载（通常 Cell 9 已加载，但做防御）
if 'model' not in dir() or model is None:
    print("加载最佳模型...")
    ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
    vocab_size = ckpt['vocab_size']
    char2idx = ckpt['char2idx']
    idx2char = ckpt['idx2char']
    state_dict = ckpt['model_state_dict']
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    model = CharRNN(vocab_size).to(DEVICE)
    model.load_state_dict(new_state_dict)
    model.eval()
    print("模型加载成功\n")

# 获取藏头字（与 Cell 9 一致）
heads = default_heads.copy()
if os.path.exists(user_input_acrostic):
    with open(user_input_acrostic, "r", encoding="utf-8") as f:
        content = f.readline().strip()
        if content and len(content) >= 4:
            heads = list(content[:4])
            print(f"使用文件提供的藏头字：{' '.join(heads)}")
        else:
            print(f"文件内容不足4字，使用默认藏头字：{' '.join(default_heads)}")
else:
    print(f"藏头字文件不存在，使用默认藏头字：{' '.join(default_heads)}")

# ---------- 1. 20 样本统计对比 ----------
strategies_20 = [
    {"name": "Greedy",         "temp": 1.0, "top_k": 1,   "top_p": None},
    {"name": "T=0.6 K=10",     "temp": 0.6, "top_k": 10,  "top_p": None},
    {"name": "T=0.8 K=20",     "temp": 0.8, "top_k": 20,  "top_p": None},
    {"name": "T=1.2 K=30",     "temp": 1.2, "top_k": 30,  "top_p": None},
    {"name": "T=0.7 P=0.9",    "temp": 0.7, "top_k": None,"top_p": 0.9},
]

print("\n" + "-" * 70)
print("每种策略生成 20 个藏头诗样本，统计平均指标")
print("-" * 70)

for s in strategies_20:
    ok_format = 0      # 格式合规（4句×7字）
    ok_first = 0       # 首字全部正确
    losses = []
    ppls = []
    for seed in range(20):
        torch.manual_seed(seed)   # 不同种子，增加多样性
        poem, token_seq = generate_acrostic(
            model, heads, char2idx, idx2char,
            temperature=s["temp"], top_k=s["top_k"], top_p=s["top_p"],
            repetition_penalty=REPETITION_PENALTY
        )
        # 格式检查
        lines = poem.strip().split('\n')
        if len(lines) == 4 and all(len(line.strip()) == 7 for line in lines):
            ok_format += 1
            # 首字检查（只有在格式合规时才检查，否则容易误判）
            if all(lines[i][0] == heads[i] for i in range(4)):
                ok_first += 1
        # 计算 loss/ppl
        loss, ppl = compute_sequence_loss_ppl(model, token_seq, char2idx)
        losses.append(loss)
        ppls.append(ppl)

    avg_loss = np.mean(losses)
    std_loss = np.std(losses)
    avg_ppl = np.mean(ppls)
    std_ppl = np.std(ppls)
    print(f"{s['name']}:")
    print(f"  格式合规率: {ok_format/20:.2%}")
    print(f"  首字命中率: {ok_first/20:.2%}")
    print(f"  平均 Loss: {avg_loss:.4f} (±{std_loss:.4f})")
    print(f"  平均 PPL: {avg_ppl:.2f} (±{std_ppl:.2f})\n")

# ---------- 2. 固定种子详细对比（展示具体诗句） ----------
print("\n" + "=" * 70)
print("各策略固定种子（seed=42）的藏头诗详细对比")
print("=" * 70)

strategies_detail = [
    {"name": "温度0.5 无top_k",   "temperature": 0.5, "top_k": None, "top_p": None},
    {"name": "温度0.7 无top_k",   "temperature": 0.7, "top_k": None, "top_p": None},
    {"name": "温度1.0 无top_k",   "temperature": 1.0, "top_k": None, "top_p": None},
    {"name": "温度1.2 无top_k",   "temperature": 1.2, "top_k": None, "top_p": None},
    {"name": "温度0.8 top_k=5",   "temperature": 0.8, "top_k": 5,   "top_p": None},
    {"name": "温度0.8 top_k=10",  "temperature": 0.8, "top_k": 10,  "top_p": None},
    {"name": "温度0.8 top_p=0.9", "temperature": 0.8, "top_k": None, "top_p": 0.9},
]

rows = []
for i, s in enumerate(strategies_detail, 1):
    torch.manual_seed(42)   # 固定种子，确保可复现
    poem, token_seq = generate_acrostic(
        model, heads, char2idx, idx2char,
        temperature=s["temperature"], top_k=s["top_k"], top_p=s["top_p"],
        repetition_penalty=REPETITION_PENALTY
    )
    loss, ppl = compute_sequence_loss_ppl(model, token_seq, char2idx)

    lines = poem.strip().split('\n')
    format_ok = (len(lines) == 4 and all(len(line.strip()) == 7 for line in lines))
    first_ok = all(lines[i][0] == heads[i] for i in range(4)) if format_ok else False

    print(f"\n【策略{i}: {s['name']}】")
    print(poem)
    print(f"Loss: {loss:.4f}, PPL: {ppl:.2f}")
    print(f"格式: {'✓' if format_ok else '✗'}, 首字正确: {'✓' if first_ok else '✗'}")

    rows.append({
        "策略": s["name"],
        "藏头诗": poem.replace('\n', ' | '),
        "Loss": f"{loss:.4f}",
        "PPL": f"{ppl:.2f}",
        "首字正确": "✓" if first_ok else "✗"
    })

# 打印汇总表格
print("\n" + "=" * 70)
print("藏头诗策略对比汇总表")
print("=" * 70)
df_acrostic = pd.DataFrame(rows)
display(HTML(df_acrostic.to_html(index=False)))


藏头诗不同采样策略对比
使用文件提供的藏头字：春 花 秋 月

----------------------------------------------------------------------
每种策略生成 20 个藏头诗样本，统计平均指标
----------------------------------------------------------------------
Greedy:
  格式合规率: 100.00%
  首字命中率: 100.00%
  平均 Loss: 2.3326 (±0.0000)
  平均 PPL: 10.30 (±0.00)

T=0.6 K=10:
  格式合规率: 100.00%
  首字命中率: 100.00%
  平均 Loss: 2.6920 (±0.1399)
  平均 PPL: 14.90 (±2.00)

T=0.8 K=20:
  格式合规率: 100.00%
  首字命中率: 100.00%
  平均 Loss: 2.9809 (±0.1687)
  平均 PPL: 20.00 (±3.69)

T=1.2 K=30:
  格式合规率: 100.00%
  首字命中率: 100.00%
  平均 Loss: 3.4612 (±0.2563)
  平均 PPL: 32.92 (±8.55)

T=0.7 P=0.9:
  格式合规率: 100.00%
  首字命中率: 100.00%
  平均 Loss: 2.9790 (±0.1457)
  平均 PPL: 19.87 (±2.80)


各策略固定种子（seed=42）的藏头诗详细对比

【策略1: 温度0.5 无top_k】
春风吹雨满江城
花落花开月正明
秋色满堂人未到
月华空绕画屏横
Loss: 2.5711, PPL: 13.08
格式: ✓, 首字正确: ✓

【策略2: 温度0.7 无top_k】
春风一夜满江干
花影无如月色寒
秋入衣裳人未起
月明空下画栏看
Loss: 3.3026, PPL: 27.18
格式: ✓, 首字正确: ✓

【策略3: 温度1.0 无top_k】
春入田家正好晴
花开花落月明明
秋温犹未收眉锦
月老风来不放声
Loss: 3.7921, PPL: 44.35
格式: ✓, 首字正确: ✓

【策略

策略,藏头诗,Loss,PPL,首字正确
温度0.5 无top_k,春风吹雨满江城 | 花落花开月正明 | 秋色满堂人未到 | 月华空绕画屏横,2.5711,13.08,✓
温度0.7 无top_k,春风一夜满江干 | 花影无如月色寒 | 秋入衣裳人未起 | 月明空下画栏看,3.3026,27.18,✓
温度1.0 无top_k,春入田家正好晴 | 花开花落月明明 | 秋温犹未收眉锦 | 月老风来不放声,3.7921,44.35,✓
温度1.2 无top_k,春忧田地正行脚 | 花讶溪桥重断魂 | 秋屋卧怜人少住 | 月明风度法成存,5.0398,154.44,✓
温度0.8 top_k=5,春风一夜满江干 | 花落花开未忍寒 | 秋色满庭人独立 | 月明空锁暮云间,2.6124,13.63,✓
温度0.8 top_k=10,春入园林绿绕篱 | 花开无数点红衣 | 秋风满地人情远 | 月照空山鸟道稀,2.8480,17.25,✓
温度0.8 top_p=0.9,春入园林绿绕篱 | 花开无数见芳菲 | 秋风似送人情远 | 月照离离不放归,3.1848,24.16,✓
